In [ ]:
# import the required libraries
import pandas as pd
from transformers import GPT2Tokenizer, GPT2Model
import torch
import numpy as np

# read the input data (cleaned embedded keywords-title dataset) into a pandas dataframe. This dataset was merged manually.
""" 
the cleaned embedded keywords-title dataset is the dataset 
formed by merging & cleaning the embedded keywords and embedded title dataset.
"""
movies_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/gpt2_chunk_3.csv")

In [ ]:
movies_df.head(10)

,adult,id,genres,imdb_id,original_language,original_title,overview,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,False,336316,Drama,tt2801494,en,Disgrace,"Milo, a rebellious young child, has been exper...",1.388,12/20/2013,English,7.3,3.0,336316.0,short film,[8],-1.31002426e-01 1.17688537e-01 -9.95410740e-0...,1.60744876e-01 -3.11993092e-01 -4.44272429e-0...
1,False,336318,"Drama, Comedy",tt0101554,en,Caught Looking,Caught Looking sees a lonely gay man attempt t...,0.912,1/1/1991,English,3.5,2.0,336318.0,gay theme,"[8, 5]",2.47805685e-01 -2.97273368e-01 -9.18112516e-0...,1.55963302e-01 2.65499294e-01 -2.86090523e-0...
2,False,336323,Comedy,tt3596382,en,Love & Taxes,Love &amp; Taxes is a riveting comic tale of s...,2.901,3/3/2017,English,4.0,1.0,336323.0,taxes monologue workplace love,[5],3.15813661e-01 6.25009357e-04 -8.07741940e-0...,5.81947528e-02 -3.08253229e-01 -1.32096663e-0...
3,False,336324,Comedy,tt0350041,en,Red Diaper Baby,Monolinguist Josh Kornbluth's meditation on gr...,0.841,4/17/2004,English,0.0,0.0,336324.0,communist monologue growing up family communism,[5],-2.28358999e-01 -8.52901582e-03 -6.82814956e-0...,4.51743007e-01 -3.33087623e-01 -4.33495015e-0...
4,False,336332,Drama,tt0326861,es,La Espera,"In Montevideo, Sonja cares for her invalid mot...",1.211,9/30/2002,Spanish,4.2,4.0,336332.0,based on novel or book,[8],1.50816038e-01 1.13374308e-01 -1.37156534e+0...,-3.61197114e-01 -2.21417293e-01 1.41059697e-0...
5,False,336337,Comedy,NaN,en,Ronny Chieng - Chieng Reaction,Pun-based comedy show names are an automatic w...,1.095,3/19/2015,English,0.0,0.0,336337.0,stand-up comedy,[5],-6.52223676e-02 1.68744475e-02 -3.78949761e-0...,-1.81439608e-01 -3.51802886e-01 -3.35585415e-0...
6,False,336342,Documentary,NaN,en,Australia's Hardest Prison: Lockdown Oz,Over the past decade Australiaâ€™s prison popu...,0.600,8/17/2008,[],6.3,3.0,336342.0,prison,[11],-2.76993543e-01 3.92419472e-02 -5.74876070e-0...,3.66676003e-02 -2.59009637e-02 1.44925609e-0...
7,False,336348,Documentary,NaN,en,Gary Barlow: Journey to Afghanistan,One of our biggest music stars Gary Barlow fac...,1.418,12/3/2013,English,0.0,0.0,336348.0,afghanistan,[11],1.32092163e-01 1.87230423e-01 -3.38446379e-0...,-1.96160510e-01 -4.27407950e-01 -1.34842053e-0...
8,False,336355,"Romance, Drama",tt1056056,tl,Endo,Leo belongs to a poor family. Because he did n...,1.400,7/21/2007,"English, Tagalog",0.0,0.0,336355.0,philippines,"[13, 8]",-4.68761891e-01 4.80118394e-02 3.34262699e-0...,2.78841779e-02 -2.14559108e-01 -2.69064754e-0...
9,False,336357,Horror,tt4379224,es,Domo,Sira works in the night shift at the Morgen Br...,0.681,3/12/2015,Spanish,2.0,1.0,336357.0,found footage security camera,[12],6.48359060e-02 7.79107735e-02 -6.19339705e-0...,2.77790576e-02 -1.77926451e-01 -1.07433891e+0...


In [ ]:
movies_df.shape[0]

46944

In [ ]:
movies_df.columns

Index(['adult', 'id', 'genres', 'imdb_id', 'original_language',
       'original_title', 'overview', 'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
drop_columns = ['adult', 'genres', 'original_language',
       'popularity', 'release_date',
       'spoken_languages', 'vote_average', 'vote_count', 'movie_id',
       'keywords', 'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview
0,336316,tt2801494,Disgrace,"Milo, a rebellious young child, has been exper..."
1,336318,tt0101554,Caught Looking,Caught Looking sees a lonely gay man attempt t...
2,336323,tt3596382,Love & Taxes,Love &amp; Taxes is a riveting comic tale of s...
3,336324,tt0350041,Red Diaper Baby,Monolinguist Josh Kornbluth's meditation on gr...
4,336332,tt0326861,La Espera,"In Montevideo, Sonja cares for her invalid mot..."


In [ ]:
# reduce the size of the overview column to 500 characters of string
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column '{}': {}".format('overview', max_length))
movies_df['overview'] = movies_df['overview'].str.slice(0, 500)
max_length = movies_df['overview'].str.len().max()
print("Maximum length of column after strip '{}': {}".format('overview', max_length))

Maximum length of column 'overview': 1022.0
Maximum length of column after strip 'overview': 500.0


In [ ]:
# load the pre-trained gpt-2 tokenizer and model
model_name='gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2Model.from_pretrained(model_name)

def create_word_embeddings(sentence):
    if pd.isna(sentence):
      return np.zeros(768)
    else:
      input_ids = tokenizer.encode(sentence, return_tensors='pt')
      with torch.no_grad():
          output = model(input_ids)
      embeddings = output.last_hidden_state.mean(dim=1).squeeze().numpy()
      return embeddings

# Apply the function to each row and create a new column 'overview_embedding'
movies_df['overview_embedding'] = movies_df['overview'].apply(create_word_embeddings)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

In [ ]:
movies_df['overview_embedding'] = movies_df['overview_embedding'].apply(lambda x: str(x))
movies_df.loc[0, 'overview_embedding']

'[-1.07935332e-01 -2.90515155e-01 -9.06704366e-01 -1.06879629e-01\n -1.73946500e-01 -4.80113059e-01  2.48419118e+00  2.70360529e-01\n -2.04504743e-01  2.58056164e-01  8.47855806e-02  1.36495903e-01\n  6.40918873e-03 -1.14537619e-01  3.47242691e-02  5.06629609e-02\n -1.75646678e-01  1.50354743e-01  1.02445811e-01 -3.12602147e-02\n  8.17657486e-02 -2.35494584e-01 -9.50753465e-02  2.11141691e-01\n  1.74240306e-01  1.87429771e-01 -3.63958180e-01  9.50331464e-02\n -2.08051652e-01  3.17097753e-02 -1.22293524e-01 -2.86526322e-01\n  1.29272938e-02 -9.77464616e-02 -2.03461125e-01  5.42541444e-01\n  6.90540314e+01 -6.15873002e-02 -2.17977926e-01  3.65499295e-02\n  1.63843840e-01 -8.30255896e-02  1.29185453e-01  7.09728058e-03\n -6.27928078e-02  5.39551750e-02 -5.82001219e-03  2.30324283e-01\n -8.98343027e-02  3.87392282e-01 -1.63068235e-01  1.77888528e-01\n -5.56383371e-01  1.43347293e-01  1.78715751e-01  1.23998773e+00\n -1.24035969e-01  2.11187024e-02  6.17827065e-02  1.88323975e-01\n -1.27491

In [ ]:
# convert the embedded column array to embedded string and clean the overview_embedding_string column and remove any commas, whitespaces and unwanted characters
movies_df['overview_embedding_string'] = movies_df['overview_embedding'].str.replace('[', '').str.replace(']', '').str.replace('\n', '').str.replace(',', '')

In [ ]:
movies_df.loc[0, 'overview_embedding_string']

'-1.07935332e-01 -2.90515155e-01 -9.06704366e-01 -1.06879629e-01 -1.73946500e-01 -4.80113059e-01  2.48419118e+00  2.70360529e-01 -2.04504743e-01  2.58056164e-01  8.47855806e-02  1.36495903e-01  6.40918873e-03 -1.14537619e-01  3.47242691e-02  5.06629609e-02 -1.75646678e-01  1.50354743e-01  1.02445811e-01 -3.12602147e-02  8.17657486e-02 -2.35494584e-01 -9.50753465e-02  2.11141691e-01  1.74240306e-01  1.87429771e-01 -3.63958180e-01  9.50331464e-02 -2.08051652e-01  3.17097753e-02 -1.22293524e-01 -2.86526322e-01  1.29272938e-02 -9.77464616e-02 -2.03461125e-01  5.42541444e-01  6.90540314e+01 -6.15873002e-02 -2.17977926e-01  3.65499295e-02  1.63843840e-01 -8.30255896e-02  1.29185453e-01  7.09728058e-03 -6.27928078e-02  5.39551750e-02 -5.82001219e-03  2.30324283e-01 -8.98343027e-02  3.87392282e-01 -1.63068235e-01  1.77888528e-01 -5.56383371e-01  1.43347293e-01  1.78715751e-01  1.23998773e+00 -1.24035969e-01  2.11187024e-02  6.17827065e-02  1.88323975e-01 -1.27491191e-01  1.59146309e-01  2.3536

In [ ]:
drop_columns = ['overview_embedding']
movies_df = movies_df.drop(columns=drop_columns)
movies_df.head(5)

,id,imdb_id,original_title,overview,overview_embedding_string
0,336316,tt2801494,Disgrace,"Milo, a rebellious young child, has been exper...",-1.07935332e-01 -2.90515155e-01 -9.06704366e-0...
1,336318,tt0101554,Caught Looking,Caught Looking sees a lonely gay man attempt t...,-1.18902080e-01 1.74247082e-02 -6.94074333e-0...
2,336323,tt3596382,Love & Taxes,Love &amp; Taxes is a riveting comic tale of s...,3.35822701e-01 1.27950951e-01 -3.33183080e-0...
3,336324,tt0350041,Red Diaper Baby,Monolinguist Josh Kornbluth's meditation on gr...,-1.08606540e-01 1.31193414e-01 -4.61276591e-0...
4,336332,tt0326861,La Espera,"In Montevideo, Sonja cares for her invalid mot...",-1.22734614e-01 9.19823647e-02 -5.87077618e-0...


In [ ]:
movies_df['overview_embedding_string'] = movies_df['overview_embedding_string'].str.strip()

In [ ]:
movies_keywords_genres = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/old_final_mkg_dataset_gpt2.csv")

In [ ]:
merged_df = pd.merge(movies_df, movies_keywords_genres, on=['id', 'imdb_id', 'original_title'], how='inner')

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,336316,tt2801494,Disgrace,"Milo, a rebellious young child, has been exper...",-1.07935332e-01 -2.90515155e-01 -9.06704366e-0...,False,Drama,en,"Milo, a rebellious young child, has been exper...",1.388,12/20/2013,English,7.3,3.0,336316.0,short film,[8],-1.31002426e-01 1.17688537e-01 -9.95410740e-0...,1.60744876e-01 -3.11993092e-01 -4.44272429e-0...
1,336318,tt0101554,Caught Looking,Caught Looking sees a lonely gay man attempt t...,-1.18902080e-01 1.74247082e-02 -6.94074333e-0...,False,"Drama, Comedy",en,Caught Looking sees a lonely gay man attempt t...,0.912,1/1/1991,English,3.5,2.0,336318.0,gay theme,"[8, 5]",2.47805685e-01 -2.97273368e-01 -9.18112516e-0...,1.55963302e-01 2.65499294e-01 -2.86090523e-0...
2,336323,tt3596382,Love & Taxes,Love &amp; Taxes is a riveting comic tale of s...,3.35822701e-01 1.27950951e-01 -3.33183080e-01...,False,Comedy,en,Love &amp; Taxes is a riveting comic tale of s...,2.901,3/3/2017,English,4.0,1.0,336323.0,taxes monologue workplace love,[5],3.15813661e-01 6.25009357e-04 -8.07741940e-0...,5.81947528e-02 -3.08253229e-01 -1.32096663e-0...
3,336324,tt0350041,Red Diaper Baby,Monolinguist Josh Kornbluth's meditation on gr...,-1.08606540e-01 1.31193414e-01 -4.61276591e-0...,False,Comedy,en,Monolinguist Josh Kornbluth's meditation on gr...,0.841,4/17/2004,English,0.0,0.0,336324.0,communist monologue growing up family communism,[5],-2.28358999e-01 -8.52901582e-03 -6.82814956e-0...,4.51743007e-01 -3.33087623e-01 -4.33495015e-0...
4,336332,tt0326861,La Espera,"In Montevideo, Sonja cares for her invalid mot...",-1.22734614e-01 9.19823647e-02 -5.87077618e-0...,False,Drama,es,"In Montevideo, Sonja cares for her invalid mot...",1.211,9/30/2002,Spanish,4.2,4.0,336332.0,based on novel or book,[8],1.50816038e-01 1.13374308e-01 -1.37156534e+0...,-3.61197114e-01 -2.21417293e-01 1.41059697e-0...


In [ ]:
merged_df.shape[0]

46944

In [ ]:
merged_df.columns

Index(['id', 'imdb_id', 'original_title', 'overview_x',
       'overview_embedding_string', 'adult', 'genres', 'original_language',
       'overview_y', 'popularity', 'release_date', 'spoken_languages',
       'vote_average', 'vote_count', 'movie_id', 'keywords', 'genres_indices',
       'keywords_embedding_string', 'original_title_embedding_string'],
      dtype='object')

In [ ]:
nan_count = merged_df['overview_embedding_string'].isnull().sum(); print(f"The column 'overview_embedding' has {nan_count} NaN values.")
nan_count = merged_df['genres'].isnull().sum(); print(f"The column 'genres' has {nan_count} NaN values.")
merged_df.dropna(subset=['overview_embedding_string'], inplace=True)
merged_df.dropna(subset=['genres'], inplace=True)

The column 'overview_embedding' has 0 NaN values.
The column 'genres' has 0 NaN values.


In [ ]:
merged_df.shape[0]

46944

In [ ]:
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/final_mkg_dataset_gpt2_chunk_3.csv', index=False)

In [ ]:
#  Test

In [ ]:
movies_df.shape[0]

46944

In [ ]:
movies_keywords_genres.shape[0]

234719

In [ ]:
merged_df.head(5)

,id,imdb_id,original_title,overview_x,overview_embedding_string,adult,genres,original_language,overview_y,popularity,release_date,spoken_languages,vote_average,vote_count,movie_id,keywords,genres_indices,keywords_embedding_string,original_title_embedding_string
0,336316,tt2801494,Disgrace,"Milo, a rebellious young child, has been exper...",-1.07935332e-01 -2.90515155e-01 -9.06704366e-0...,False,Drama,en,"Milo, a rebellious young child, has been exper...",1.388,12/20/2013,English,7.3,3.0,336316.0,short film,[8],-1.31002426e-01 1.17688537e-01 -9.95410740e-0...,1.60744876e-01 -3.11993092e-01 -4.44272429e-0...
1,336318,tt0101554,Caught Looking,Caught Looking sees a lonely gay man attempt t...,-1.18902080e-01 1.74247082e-02 -6.94074333e-0...,False,"Drama, Comedy",en,Caught Looking sees a lonely gay man attempt t...,0.912,1/1/1991,English,3.5,2.0,336318.0,gay theme,"[8, 5]",2.47805685e-01 -2.97273368e-01 -9.18112516e-0...,1.55963302e-01 2.65499294e-01 -2.86090523e-0...
2,336323,tt3596382,Love & Taxes,Love &amp; Taxes is a riveting comic tale of s...,3.35822701e-01 1.27950951e-01 -3.33183080e-01...,False,Comedy,en,Love &amp; Taxes is a riveting comic tale of s...,2.901,3/3/2017,English,4.0,1.0,336323.0,taxes monologue workplace love,[5],3.15813661e-01 6.25009357e-04 -8.07741940e-0...,5.81947528e-02 -3.08253229e-01 -1.32096663e-0...
3,336324,tt0350041,Red Diaper Baby,Monolinguist Josh Kornbluth's meditation on gr...,-1.08606540e-01 1.31193414e-01 -4.61276591e-0...,False,Comedy,en,Monolinguist Josh Kornbluth's meditation on gr...,0.841,4/17/2004,English,0.0,0.0,336324.0,communist monologue growing up family communism,[5],-2.28358999e-01 -8.52901582e-03 -6.82814956e-0...,4.51743007e-01 -3.33087623e-01 -4.33495015e-0...
4,336332,tt0326861,La Espera,"In Montevideo, Sonja cares for her invalid mot...",-1.22734614e-01 9.19823647e-02 -5.87077618e-0...,False,Drama,es,"In Montevideo, Sonja cares for her invalid mot...",1.211,9/30/2002,Spanish,4.2,4.0,336332.0,based on novel or book,[8],1.50816038e-01 1.13374308e-01 -1.37156534e+0...,-3.61197114e-01 -2.21417293e-01 1.41059697e-0...


In [ ]:
# merge the embedded keywords, title and overview columns to form the input gpt2 embedded dataset chunk-3
drop_columns = ['imdb_id', 'original_title', 'overview_x', 'adult', 'original_language', 'overview_y', 'popularity', 'spoken_languages', 'vote_average', 'vote_count', 'movie_id', 'keywords']
merged_df = merged_df.drop(columns=drop_columns)
merged_df.head(5)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,336316,-1.07935332e-01 -2.90515155e-01 -9.06704366e-0...,Drama,12/20/2013,[8],-1.31002426e-01 1.17688537e-01 -9.95410740e-0...,1.60744876e-01 -3.11993092e-01 -4.44272429e-0...
1,336318,-1.18902080e-01 1.74247082e-02 -6.94074333e-0...,"Drama, Comedy",1/1/1991,"[8, 5]",2.47805685e-01 -2.97273368e-01 -9.18112516e-0...,1.55963302e-01 2.65499294e-01 -2.86090523e-0...
2,336323,3.35822701e-01 1.27950951e-01 -3.33183080e-01...,Comedy,3/3/2017,[5],3.15813661e-01 6.25009357e-04 -8.07741940e-0...,5.81947528e-02 -3.08253229e-01 -1.32096663e-0...
3,336324,-1.08606540e-01 1.31193414e-01 -4.61276591e-0...,Comedy,4/17/2004,[5],-2.28358999e-01 -8.52901582e-03 -6.82814956e-0...,4.51743007e-01 -3.33087623e-01 -4.33495015e-0...
4,336332,-1.22734614e-01 9.19823647e-02 -5.87077618e-0...,Drama,9/30/2002,[8],1.50816038e-01 1.13374308e-01 -1.37156534e+0...,-3.61197114e-01 -2.21417293e-01 1.41059697e-0...


In [ ]:
merged_df.shape[0]

46944

In [ ]:
# export the embedded keywords, title and overview dataset (chunk-3)
merged_df.to_csv('/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/f-final_mkg_dataset_gpt2_chunk_3.csv', index=False)

In [ ]:
# Testing read

In [ ]:
test_df = pd.read_csv("/content/drive/MyDrive/LLM-NEW-2/gpt2-embed/input/Chunks/f-final_mkg_dataset_gpt2_chunk_3.csv")

In [ ]:
test_df.shape[0]

46944

In [ ]:
test_df.columns

Index(['id', 'overview_embedding_string', 'genres', 'release_date',
       'genres_indices', 'keywords_embedding_string',
       'original_title_embedding_string'],
      dtype='object')

In [ ]:
test_df.head(4)

,id,overview_embedding_string,genres,release_date,genres_indices,keywords_embedding_string,original_title_embedding_string
0,336316,-1.07935332e-01 -2.90515155e-01 -9.06704366e-0...,Drama,12/20/2013,[8],-1.31002426e-01 1.17688537e-01 -9.95410740e-0...,1.60744876e-01 -3.11993092e-01 -4.44272429e-0...
1,336318,-1.18902080e-01 1.74247082e-02 -6.94074333e-0...,"Drama, Comedy",1/1/1991,"[8, 5]",2.47805685e-01 -2.97273368e-01 -9.18112516e-0...,1.55963302e-01 2.65499294e-01 -2.86090523e-0...
2,336323,3.35822701e-01 1.27950951e-01 -3.33183080e-01...,Comedy,3/3/2017,[5],3.15813661e-01 6.25009357e-04 -8.07741940e-0...,5.81947528e-02 -3.08253229e-01 -1.32096663e-0...
3,336324,-1.08606540e-01 1.31193414e-01 -4.61276591e-0...,Comedy,4/17/2004,[5],-2.28358999e-01 -8.52901582e-03 -6.82814956e-0...,4.51743007e-01 -3.33087623e-01 -4.33495015e-0...
